# Real Data Performance

This notebook compares the performance of the EM approach to LOOCV approach using real-world datasets.

## Preview Experiment

In [ ]:
import numpy as np
import pandas as pd

from fastridge import RidgeEM, RidgeLOOCV
from experiments import EmpiricalDataExperiment
from problems import EmpiricalDataProblem

problems = [
    EmpiricalDataProblem('abalone',    'Rings'),
    EmpiricalDataProblem('airfoil',    'scaled-sound-pressure'),
    EmpiricalDataProblem('concrete',   'Concrete compressive strength'),
    EmpiricalDataProblem('diabetes',   'target'),
    EmpiricalDataProblem('eye',        'y'),
    EmpiricalDataProblem('forest',     'area'),
    EmpiricalDataProblem('student',    'G3', drop=['G1', 'G2']),
    EmpiricalDataProblem('yacht',      'Residuary_resistance'),
    EmpiricalDataProblem('automobile', 'price', nan_policy='drop_rows'),
]

estimators = {
    'EM':     RidgeEM(),
    'CV_fix': RidgeLOOCV(alphas=np.logspace(-10, 10, 100, endpoint=True, base=10)),
    'CV_glm': RidgeLOOCV(alphas=100),
}

exp = EmpiricalDataExperiment(
    problems, list(estimators.values()),
    n_iterations=10, seed=123,
    est_names=list(estimators.keys())).run()
print()

In [ ]:
def stat_mean(exp, stat_name, est_name, problem_idx):
    j = exp.est_names.index(est_name)
    return np.nanmean(getattr(exp, stat_name + '_')[:, problem_idx, 0, j])

rows = []
for i, problem in enumerate(exp.problems):
    em_time = stat_mean(exp, 'fitting_time', 'EM', i)
    cv_time = (stat_mean(exp, 'fitting_time', 'CV_glm', i)
               + stat_mean(exp, 'fitting_time', 'CV_fix', i)) / 2
    row = {'dataset': problem.dataset, 'target': problem.target}
    row.update({est: stat_mean(exp, 'prediction_r2', est, i) for est in exp.est_names})
    row['Speed-Up'] = cv_time / em_time
    row['p']        = stat_mean(exp, 'number_of_features', 'EM', i)
    row['n_train']  = int(exp.ns[i, 0])
    row['n:p']      = int(exp.ns[i, 0]) / stat_mean(exp, 'number_of_features', 'EM', i)
    rows.append(row)
pd.DataFrame(rows).sort_values('n_train', ascending=False).round(2)

In [3]:
problems_d2 = [
    EmpiricalDataProblem('abalone',    'Rings'),
    EmpiricalDataProblem('airfoil',    'scaled-sound-pressure'),
    EmpiricalDataProblem('concrete',   'Concrete compressive strength'),
    EmpiricalDataProblem('diabetes',   'target'),
    EmpiricalDataProblem('eye',        'y'),
    EmpiricalDataProblem('forest',     'area'),  # OHE interaction columns cause SVD failure
    EmpiricalDataProblem('student',    'G3', drop=['G1', 'G2']),
    EmpiricalDataProblem('yacht',      'Residuary_resistance'),
    EmpiricalDataProblem('automobile', 'price', nan_policy='drop_rows'),
]

In [ ]:
exp_d2 = EmpiricalDataExperiment(
    problems_d2, list(estimators.values()),
    n_iterations=10, seed=123, polynomial=2,
    est_names=list(estimators.keys())).run()
print()

In [ ]:
rows_d2 = []
for i, problem in enumerate(exp_d2.problems):
    em_time = stat_mean(exp_d2, 'fitting_time', 'EM', i)
    cv_time = (stat_mean(exp_d2, 'fitting_time', 'CV_glm', i)
               + stat_mean(exp_d2, 'fitting_time', 'CV_fix', i)) / 2
    row = {'dataset': problem.dataset, 'target': problem.target}
    row.update({est: stat_mean(exp_d2, 'prediction_r2', est, i) for est in exp_d2.est_names})
    row['Speed-Up'] = cv_time / em_time
    row['p']        = stat_mean(exp_d2, 'number_of_features', 'EM', i)
    row['n_train']  = int(exp_d2.ns[i, 0])
    row['n:p']      = int(exp_d2.ns[i, 0]) / stat_mean(exp_d2, 'number_of_features', 'EM', i)
    rows_d2.append(row)
pd.DataFrame(rows_d2).sort_values('n_train', ascending=False).round(2)

In [6]:
problems_d3 = [
    EmpiricalDataProblem('abalone',    'Rings'),
    EmpiricalDataProblem('airfoil',    'scaled-sound-pressure'),
    EmpiricalDataProblem('concrete',   'Concrete compressive strength'),
    EmpiricalDataProblem('diabetes',   'target'),
    # EmpiricalDataProblem('eye',        'y'),  # excluded in paper for d=3
    # EmpiricalDataProblem('forest',     'area'),  # OHE interaction columns cause SVD failure
    EmpiricalDataProblem('student',    'G3', drop=['G1', 'G2']),
    EmpiricalDataProblem('yacht',      'Residuary_resistance'),
    EmpiricalDataProblem('automobile', 'price', nan_policy='drop_rows'),
]

In [ ]:
exp_d3 = EmpiricalDataExperiment(
    problems_d3, list(estimators.values()),
    n_iterations=10, seed=123, polynomial=3,
    est_names=list(estimators.keys())).run()
print()

In [ ]:
rows_d3 = []
for i, problem in enumerate(exp_d3.problems):
    em_time = stat_mean(exp_d3, 'fitting_time', 'EM', i)
    cv_time = (stat_mean(exp_d3, 'fitting_time', 'CV_glm', i)
               + stat_mean(exp_d3, 'fitting_time', 'CV_fix', i)) / 2
    row = {'dataset': problem.dataset, 'target': problem.target}
    row.update({est: stat_mean(exp_d3, 'prediction_r2', est, i) for est in exp_d3.est_names})
    row['Speed-Up'] = cv_time / em_time
    row['p']        = stat_mean(exp_d3, 'number_of_features', 'EM', i)
    row['n_train']  = int(exp_d3.ns[i, 0])
    row['n:p']      = int(exp_d3.ns[i, 0]) / stat_mean(exp_d3, 'number_of_features', 'EM', i)
    rows_d3.append(row)
pd.DataFrame(rows_d3).sort_values('n_train', ascending=False).round(2)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def make_figure3(exp_d1, exp_d2, exp_d3, output_path=None):
    """2x3 scatter of EM vs CV R² for d=1,2,3. Color = speed-up ratio.

    Top row: CV_glm on y-axis. Bottom row: CV_fix on y-axis.
    Values below CLIP_MIN=-0.1 are clipped to CLIP_MIN and shown with a dashed edge.
    Axis limits extend PAD beyond [CLIP_MIN, 1] so boundary points are fully visible.

    Each argument may be a single EmpiricalDataExperiment or a list of them
    (use a list to concatenate results from multiple experiments for the same degree).
    """
    def _as_list(e):
        return e if isinstance(e, list) else [e]

    def _r2(exps, est_name):
        result = []
        for exp in _as_list(exps):
            j = exp.est_names.index(est_name)
            result.extend(np.nanmean(exp.prediction_r2_[:, :, 0, j], axis=0).tolist())
        return result

    def _time(exps, est_name):
        result = []
        for exp in _as_list(exps):
            j = exp.est_names.index(est_name)
            result.extend(np.nanmean(exp.fitting_time_[:, :, 0, j], axis=0).tolist())
        return result

    experiments = [exp_d1, exp_d2, exp_d3]
    cv_rows = [('CV_glm', 'CV GLM Grid'), ('CV_fix', 'CV Fixed Grid')]
    CLIP_MIN = -0.1
    PAD      =  0.03

    all_su = [
        t_cv / t_em
        for exps in experiments
        for cv, _ in cv_rows
        for t_cv, t_em in zip(_time(exps, cv), _time(exps, "EM"))
    ]
    norm = mcolors.Normalize(vmin=min(all_su), vmax=max(all_su))
    cmap = plt.cm.Greens

    fig, axes = plt.subplots(2, 3, figsize=(9, 5.4), sharex=True, sharey=True)
    fig.subplots_adjust(left=0.11, right=0.84, bottom=0.11, top=0.93,
                        hspace=0.06, wspace=0.04)

    for col, exps in enumerate(experiments):
        for row, (cv_name, cv_label) in enumerate(cv_rows):
            ax = axes[row, col]

            true_em = _r2(exps, "EM")
            true_cv = _r2(exps, cv_name)
            su      = [t_cv / t_em
                       for t_cv, t_em in zip(_time(exps, cv_name), _time(exps, "EM"))]
            disp_em = [max(CLIP_MIN, x) for x in true_em]
            disp_cv = [max(CLIP_MIN, y) for y in true_cv]
            clipped = [e < CLIP_MIN or c < CLIP_MIN
                       for e, c in zip(true_em, true_cv)]
            colors  = cmap(norm(np.array(su)))

            idx_in  = [i for i, cl in enumerate(clipped) if not cl]
            idx_out = [i for i, cl in enumerate(clipped) if cl]

            if idx_in:
                ax.scatter([disp_em[i] for i in idx_in],
                           [disp_cv[i] for i in idx_in],
                           c=[colors[i] for i in idx_in],
                           s=50, zorder=3, edgecolors="k", linewidths=0.6)
            if idx_out:
                sc = ax.scatter([disp_em[i] for i in idx_out],
                                [disp_cv[i] for i in idx_out],
                                c=[colors[i] for i in idx_out],
                                s=50, zorder=4, edgecolors="k", linewidths=0.8)
                sc.set_linestyle("--")

            ax.plot([CLIP_MIN, 1], [CLIP_MIN, 1], "k--", lw=0.8, zorder=2)
            ax.axhline(0, color="0.8", lw=0.5, zorder=1)
            ax.axvline(0, color="0.8", lw=0.5, zorder=1)
            ax.set_xlim(CLIP_MIN - PAD, 1 + PAD)
            ax.set_ylim(CLIP_MIN - PAD, 1 + PAD)

            if row == 1:
                ax.set_xlabel('BayesEM ^2$')
                ax.set_xticks([0.0, 0.5, 1.0])
            if col == 0:
                ax.set_ylabel(f'{cv_label} ^2$')
                ax.set_yticks([0.0, 0.5, 1.0])
            if row == 0:
                ax.set_title(f' = {col + 1}$')

    cbar_ax = fig.add_axes([0.86, 0.28, 0.02, 0.46])
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.ax.yaxis.set_label_position("right")
    cbar.set_label('speed-up ratio', rotation=90, labelpad=-30)

    if output_path:
        fig.savefig(output_path, bbox_inches='tight')

In [ ]:
make_figure3(exp, exp_d2, exp_d3)

## Full Experiment: Small and Moderately-sized Datasets

This experiment corresponds to Figure 3 of the appendix.

In [ ]:
problems_full = [
    EmpiricalDataProblem('abalone',          'Rings'),
    EmpiricalDataProblem('airfoil',          'scaled-sound-pressure'),
    EmpiricalDataProblem('automobile',       'price',               nan_policy='drop_rows'),
    EmpiricalDataProblem('autompg',          'mpg',
                         drop=['car_name'], nan_policy='drop_rows'), # could include brand part of car name for increasing p
    EmpiricalDataProblem('crime',            'ViolentCrimesPerPop',
                         drop=['state', 'fold', 'communityname'],
                         nan_policy='drop_cols'),
    EmpiricalDataProblem('ribo',             'y'),
    EmpiricalDataProblem('eye',              'y'),
    EmpiricalDataProblem('boston',           'medv'),
    EmpiricalDataProblem('concrete',         'Concrete compressive strength'),
    EmpiricalDataProblem('diabetes',         'target'),
    EmpiricalDataProblem('facebook',         'Total Interactions',
                         drop=['comment', 'like', 'share'],
                         nan_policy='drop_rows'),  
    EmpiricalDataProblem('forest',           'area'),
    EmpiricalDataProblem('naval_propulsion', 'GT_compressor_decay', drop=['GT_turbine_decay']),  # 'Ts', 'Tp' could be extra targets, but dropping reduces r2
    EmpiricalDataProblem('naval_propulsion', 'GT_turbine_decay',    drop=['GT_compressor_decay']), # 'Ts', 'Tp' could be extra targets, but dropping reduces r2
    EmpiricalDataProblem('parkinsons',       'motor_UPDRS',         drop=['total_UPDRS']),
    EmpiricalDataProblem('parkinsons',       'total_UPDRS',         drop=['motor_UPDRS']),
    EmpiricalDataProblem('real_estate',      'Y house price of unit area'),
    EmpiricalDataProblem('student',          'G3',                  drop=['G1', 'G2']),
    EmpiricalDataProblem('yacht',            'Residuary_resistance'),
]

estimators_full = {
    'EM':     RidgeEM(),
    'CV_fix': RidgeLOOCV(alphas=np.logspace(-10, 10, 100, endpoint=True, base=10)),
    'CV_glm': RidgeLOOCV(alphas=100),
}

exp_full = EmpiricalDataExperiment(
    problems_full, list(estimators_full.values()),
    n_iterations=100, seed=123,
    est_names=list(estimators_full.keys())).run()
print()

In [ ]:
rows_full = []
for i, problem in enumerate(exp_full.problems):
    em_time = stat_mean(exp_full, 'fitting_time', 'EM', i)
    cv_time = (stat_mean(exp_full, 'fitting_time', 'CV_glm', i)
               + stat_mean(exp_full, 'fitting_time', 'CV_fix', i)) / 2
    row = {'dataset': problem.dataset, 'target': problem.target}
    row.update({est: stat_mean(exp_full, 'prediction_r2', est, i) for est in exp_full.est_names})
    row['Speed Up Ratio'] = cv_time / em_time
    row['p']              = stat_mean(exp_full, 'number_of_features', 'EM', i)
    row['n_train']        = int(exp_full.ns[i, 0])
    row['n:p']            = int(exp_full.ns[i, 0]) / stat_mean(exp_full, 'number_of_features', 'EM', i)
    rows_full.append(row)
pd.DataFrame(rows_full).sort_values('n_train', ascending=False).round(2)

In [14]:
problems_full_d2 = [
    EmpiricalDataProblem('abalone',          'Rings'),
    EmpiricalDataProblem('airfoil',          'scaled-sound-pressure'),
    EmpiricalDataProblem('automobile',       'price',               nan_policy='drop_rows'),
    EmpiricalDataProblem('autompg',          'mpg',
                         drop=['car_name'], nan_policy='drop_rows'),
    EmpiricalDataProblem('crime',            'ViolentCrimesPerPop',
                         drop=['state', 'fold', 'communityname'],
                         nan_policy='drop_cols'),
    # EmpiricalDataProblem('ribo',             'y'),  # memory exhaustion at d=2
    EmpiricalDataProblem('eye',              'y'),
    EmpiricalDataProblem('boston',           'medv'),
    EmpiricalDataProblem('concrete',         'Concrete compressive strength'),
    EmpiricalDataProblem('diabetes',         'target'),
    EmpiricalDataProblem('facebook',         'Total Interactions',
                         drop=['comment', 'like', 'share'],
                         nan_policy='drop_rows'),
    # EmpiricalDataProblem('forest',           'area'),  # OHE interaction columns cause SVD failure
    EmpiricalDataProblem('naval_propulsion', 'GT_compressor_decay', drop=['GT_turbine_decay']),
    EmpiricalDataProblem('naval_propulsion', 'GT_turbine_decay',    drop=['GT_compressor_decay']),
    EmpiricalDataProblem('parkinsons',       'motor_UPDRS',         drop=['total_UPDRS']),
    EmpiricalDataProblem('parkinsons',       'total_UPDRS',         drop=['motor_UPDRS']),
    EmpiricalDataProblem('real_estate',      'Y house price of unit area'),
    EmpiricalDataProblem('student',          'G3',                  drop=['G1', 'G2']),
    EmpiricalDataProblem('yacht',            'Residuary_resistance'),
]

In [ ]:
exp_full_d2 = EmpiricalDataExperiment(
    problems_full_d2, list(estimators_full.values()),
    n_iterations=30, seed=123, polynomial=2,
    est_names=list(estimators_full.keys())).run()
print()

In [ ]:
rows_full_d2 = []
for i, problem in enumerate(exp_full_d2.problems):
    em_time = stat_mean(exp_full_d2, 'fitting_time', 'EM', i)
    cv_time = (stat_mean(exp_full_d2, 'fitting_time', 'CV_glm', i)
               + stat_mean(exp_full_d2, 'fitting_time', 'CV_fix', i)) / 2
    row = {'dataset': problem.dataset, 'target': problem.target}
    row.update({est: stat_mean(exp_full_d2, 'prediction_r2', est, i) for est in exp_full_d2.est_names})
    row['Speed Up Ratio'] = cv_time / em_time
    row['p']              = stat_mean(exp_full_d2, 'number_of_features', 'EM', i)
    row['n_train']        = int(exp_full_d2.ns[i, 0])
    row['n:p']            = int(exp_full_d2.ns[i, 0]) / stat_mean(exp_full_d2, 'number_of_features', 'EM', i)
    rows_full_d2.append(row)
pd.DataFrame(rows_full_d2).sort_values('n_train', ascending=False).round(2)

In [17]:
problems_full_d3 = [
    EmpiricalDataProblem('abalone',          'Rings'),
    EmpiricalDataProblem('airfoil',          'scaled-sound-pressure'),
    EmpiricalDataProblem('automobile',       'price',               nan_policy='drop_rows'),
    EmpiricalDataProblem('autompg',          'mpg',
                         drop=['car_name'], nan_policy='drop_rows'),
    EmpiricalDataProblem('crime',            'ViolentCrimesPerPop',
                         drop=['state', 'fold', 'communityname'],
                         nan_policy='drop_cols'),
    # EmpiricalDataProblem('ribo',             'y'),  # memory exhaustion at d>=2
    # EmpiricalDataProblem('eye',              'y'),  # excluded in paper for d=3
    EmpiricalDataProblem('boston',           'medv'),
    EmpiricalDataProblem('concrete',         'Concrete compressive strength'),
    EmpiricalDataProblem('diabetes',         'target'),
    EmpiricalDataProblem('facebook',         'Total Interactions',
                         drop=['comment', 'like', 'share'],
                         nan_policy='drop_rows'),
    # EmpiricalDataProblem('forest',           'area'),  # OHE interaction columns cause SVD failure
    EmpiricalDataProblem('naval_propulsion', 'GT_compressor_decay', drop=['GT_turbine_decay']),
    EmpiricalDataProblem('naval_propulsion', 'GT_turbine_decay',    drop=['GT_compressor_decay']),
    EmpiricalDataProblem('parkinsons',       'motor_UPDRS',         drop=['total_UPDRS']),
    EmpiricalDataProblem('parkinsons',       'total_UPDRS',         drop=['motor_UPDRS']),
    EmpiricalDataProblem('real_estate',      'Y house price of unit area'),
    EmpiricalDataProblem('student',          'G3',                  drop=['G1', 'G2']),
    EmpiricalDataProblem('yacht',            'Residuary_resistance'),
]

In [ ]:
exp_full_d3 = EmpiricalDataExperiment(
    problems_full_d3, list(estimators_full.values()),
    n_iterations=30, seed=123, polynomial=3,
    est_names=list(estimators_full.keys())).run()
print()

In [ ]:
rows_full_d3 = []
for i, problem in enumerate(exp_full_d3.problems):
    em_time = stat_mean(exp_full_d3, 'fitting_time', 'EM', i)
    cv_time = (stat_mean(exp_full_d3, 'fitting_time', 'CV_glm', i)
               + stat_mean(exp_full_d3, 'fitting_time', 'CV_fix', i)) / 2
    row = {'dataset': problem.dataset, 'target': problem.target}
    row.update({est: stat_mean(exp_full_d3, 'prediction_r2', est, i) for est in exp_full_d3.est_names})
    row['Speed Up Ratio'] = cv_time / em_time
    row['p']              = stat_mean(exp_full_d3, 'number_of_features', 'EM', i)
    row['n_train']        = int(exp_full_d3.ns[i, 0])
    row['n:p']            = int(exp_full_d3.ns[i, 0]) / stat_mean(exp_full_d3, 'number_of_features', 'EM', i)
    rows_full_d3.append(row)
pd.DataFrame(rows_full_d3).sort_values('n_train', ascending=False).round(2)

In [ ]:
make_figure3(exp_full, exp_full_d2, exp_full_d3,
             output_path='../output/paper2023_figure3.pdf')

In [21]:
problems_large = [
    EmpiricalDataProblem('twitter', 'V78'),
    EmpiricalDataProblem('tomshw', 'V97'),
    EmpiricalDataProblem('blog',   'V281'),
    EmpiricalDataProblem('ct_slices', 'reference'),
]

In [ ]:
exp_large = EmpiricalDataExperiment(
    problems_large, list(estimators_full.values()),
    n_iterations=30, seed=123,
    est_names=list(estimators_full.keys())).run()
print()

In [ ]:
rows_large = []
for i, problem in enumerate(exp_large.problems):
    em_time = stat_mean(exp_large, 'fitting_time', 'EM', i)
    cv_time = (stat_mean(exp_large, 'fitting_time', 'CV_glm', i)
               + stat_mean(exp_large, 'fitting_time', 'CV_fix', i)) / 2
    row = {'dataset': problem.dataset, 'target': problem.target}
    row.update({est: stat_mean(exp_large, 'prediction_r2', est, i) for est in exp_large.est_names})
    row['Speed Up Ratio'] = cv_time / em_time
    row['p']              = stat_mean(exp_large, 'number_of_features', 'EM', i)
    row['n_train']        = int(exp_large.ns[i, 0])
    row['n:p']            = int(exp_large.ns[i, 0]) / stat_mean(exp_large, 'number_of_features', 'EM', i)
    rows_large.append(row)
pd.DataFrame(rows_large).sort_values('n_train', ascending=False).round(2)

In [24]:
problems_large_d2 = [
    EmpiricalDataProblem('twitter', 'V78'),
    EmpiricalDataProblem('tomshw', 'V97'),
    EmpiricalDataProblem('blog',   'V281'),
    EmpiricalDataProblem('ct_slices', 'reference'),
]

In [ ]:
exp_large_d2 = EmpiricalDataExperiment(
    problems_large_d2, list(estimators_full.values()),
    n_iterations=30, seed=123, polynomial=2,
    est_names=list(estimators_full.keys())).run()
print()

In [ ]:
rows_large_d2 = []
for i, problem in enumerate(exp_large_d2.problems):
    em_time = stat_mean(exp_large_d2, 'fitting_time', 'EM', i)
    cv_time = (stat_mean(exp_large_d2, 'fitting_time', 'CV_glm', i)
               + stat_mean(exp_large_d2, 'fitting_time', 'CV_fix', i)) / 2
    row = {'dataset': problem.dataset, 'target': problem.target}
    row.update({est: stat_mean(exp_large_d2, 'prediction_r2', est, i) for est in exp_large_d2.est_names})
    row['Speed Up Ratio'] = cv_time / em_time
    row['p']              = stat_mean(exp_large_d2, 'number_of_features', 'EM', i)
    row['n_train']        = int(exp_large_d2.ns[i, 0])
    row['n:p']            = int(exp_large_d2.ns[i, 0]) / stat_mean(exp_large_d2, 'number_of_features', 'EM', i)
    rows_large_d2.append(row)
pd.DataFrame(rows_large_d2).sort_values('n_train', ascending=False).round(2)

In [ ]:
make_figure3([exp_full, exp_large],
             [exp_full_d2, exp_large_d2],
             exp_full_d3,
             output_path='../output/realdata_r2_by_degree.pdf')